In [1]:
import torch, torch_geometric

device = torch.device("mps" if torch.backends.mps.is_available() else print("CPU"))

/Users/krishpatel/code/VSC/HOMO_LUMO_Prediction/gnn/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import os
import time
from collections import Counter

import torch
from torch import nn
from torch.nn import functional as F
import matplotlib.pyplot as plt

from torch_geometric.datasets import QM9
from torch_geometric.data import Data, Batch
from torch_geometric.loader import DataLoader
from torch_geometric.nn import MessagePassing, global_mean_pool, global_add_pool

torch.manual_seed(0)

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("torch:", torch.__version__)
print("device:", device)

torch: 2.13.0
device: mps


In [3]:
torch.manual_seed(0)
a = torch.randn(3)
b = torch.randn(3)
print("a first: ", a.tolist())
print("b second:", b.tolist())

torch.manual_seed(0)
b2 = torch.randn(3)
a2 = torch.randn(3)
print("b first: ", b2.tolist())
print("a second:", a2.tolist())

a first:  [1.5409960746765137, -0.293428897857666, -2.1787893772125244]
b second: [0.5684312582015991, -1.0845223665237427, -1.3985954523086548]
b first:  [1.5409960746765137, -0.293428897857666, -2.1787893772125244]
a second: [0.5684312582015991, -1.0845223665237427, -1.3985954523086548]


In [4]:
import sys
print(sys.executable)

/Users/krishpatel/code/VSC/HOMO_LUMO_Prediction/gnn/bin/python


In [ ]:
#loading the QM9 dataset (to only utilize row 4) and verifying the number of molecules and the target vector per molecule
dataset = QM9(root='data/QM9')

print(dataset)
print("molecules:", len(dataset))
print("target vector per molecule:", tuple(dataset[0].y.shape))

Processing...
Using a pre-processed version of the dataset. Please install 'rdkit' to alternatively process the raw data.
Done!


QM9(130831)
molecules: 130831
target vector per molecule: (1, 19)


In [7]:
#verifying that row 4 is row 3-2 (homo-lumo eV size)

TARGET = 4
ATOM_TYPES = ["H", "C", "N", "O", "F"]
BOND_TYPES = ["single", "double", "triple", "aromatic"]

y = dataset[0].y[0]
print("HOMO:", round(y[2].item(), 4), "eV")
print("LUMO:", round(y[3].item(), 4), "eV")
print("gap :", round(y[4].item(), 4), "eV")
print("LUMO - HOMO =", round((y[3] - y[2]).item(), 4), "eV")

HOMO: -10.5499 eV
LUMO: 3.1865 eV
gap : 13.7363 eV
LUMO - HOMO = 13.7363 eV
